In [1]:


# %%
import re
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = BASE_DIR / "data" / "raw" / "Database-Fe-Ni-Co-Al-Cr-Ti.xlsx"
FILLED_PATH = BASE_DIR / "data" / "processed" / "Database-Fe-Ni-Co-Al-Cr-Ti_filled.xlsx"
STATS_PATH = BASE_DIR / "outputs" / "tables" / "data_audit_statistics.xlsx"
for p in (FILLED_PATH, STATS_PATH):
    p.parent.mkdir(parents=True, exist_ok=True)

ELEMENTS = ['Ni', 'Al', 'Co', 'Cr', 'Cu', 'Fe', 'Ti',
            'V', 'Mo', 'Hf', 'W', 'Ta', 'B', 'C', 'Nb', 'Mn']
ZERO_COLS = ["HT 1, °C", "HT 1, min", "Recrystallization, °C", "Recrystallization, min",
             "Aging 1, °C", "Aging 1, min", "Aging 2, °C", "Aging 2, min",
             "Cold deformation, %"]
NAN_COLS_NUM = ["Solvus Temperature, °C", "L12 fraction, %"]
NAN_COLS_TXT = ["Cooling_after_HT1", "Cooling_after_Recryst", "Cooling_after_Aging1"]
NAN_COLS = NAN_COLS_NUM + NAN_COLS_TXT

# %%
# Загрузка
df = pd.read_excel(SRC)
df.columns = [c.strip() for c in df.columns]
needed = (ELEMENTS + ZERO_COLS + NAN_COLS +
          ['Paper', 'DOI', 'Phase composition', 'Target state', 'L12 presence'])
missing = [c for c in needed if c not in df.columns]
assert not missing, f"Нет колонок: {missing}\nДоступные: {list(df.columns)}"

df = df.replace(r"^\s*$", np.nan, regex=True)


In [2]:
# %%
# %%
# 2. Проверка фазовых меток (чистка выполнена в самой базе)
phase = df['Phase composition']

ALLOWED_GREEK = 'ητσ'

phase = df['Phase composition']
assert phase.notna().all(), "Есть записи без фазового состава"
assert not phase.str.contains(rf'[^\x00-\x7F{ALLOWED_GREEK}]').any(), \
    "Есть посторонние символы вне ASCII и разрешённых греческих букв"
assert (phase == phase.str.strip()).all(), "Есть пробелы по краям метки"
assert not phase.str.contains(r'\s{2,}|(?<! )\+|\+(?! )').any(), "Нарушен формат разделителя ' + '"

tokens = phase.str.split(' + ', regex=False).explode()
print(f"Уникальных комбинаций фаз: {phase.nunique()}, отдельных фаз: {tokens.nunique()}")
print(tokens.value_counts())

# %%
# 3. Концентрации: проверка колонок и значений, пропуски -> 0
missing = [el for el in ELEMENTS if el not in df.columns]
assert not missing, f"Нет колонок элементов: {missing}"

for el in ELEMENTS:
    num = pd.to_numeric(df[el], errors='coerce')
    bad = df[el].notna() & num.isna()
    assert not bad.any(), f"{el}: нечисловые значения в строках {list(df.index[bad])}"
    df[el] = num.fillna(0.0)

Уникальных комбинаций фаз: 32, отдельных фаз: 17
Phase composition
FCC              360
L12              281
B2                55
L21               43
η                 26
BCC               18
IM                18
σ                  8
D019               3
Laves              2
Cr23C6             1
other              1
Al-rich phase      1
carbides           1
D022               1
τ                  1
borides            1
Name: count, dtype: int64


In [3]:
# %%
# Статистика ДО заполнения
el_num = df[ELEMENTS].apply(pd.to_numeric, errors='coerce')
assert not (df[ELEMENTS].notna() & el_num.isna()).any().any(), "Нечисловые значения в элементах"
pr_num = df[ZERO_COLS].apply(pd.to_numeric, errors='coerce')
assert not (df[ZERO_COLS].notna() & pr_num.isna()).any().any(), "Нечисловые значения в параметрах ТО"
opt_num = df[NAN_COLS].apply(pd.to_numeric, errors='coerce')

summary = pd.DataFrame({
    'Metric': ['Records', 'Columns', 'Unique papers', 'Unique DOI'],
    'Value': [len(df), df.shape[1], df['Paper'].nunique(), df['DOI'].nunique()]})

cols_summary = pd.DataFrame({
    'Column': df.columns,
    'dtype': [str(df[c].dtype) for c in df.columns],
    'Filled': [int(df[c].notna().sum()) for c in df.columns],
    'Empty': [int(df[c].isna().sum()) for c in df.columns],
    'Filled, %': [round(df[c].notna().mean() * 100, 1) for c in df.columns],
    'Unique values': [int(df[c].nunique()) for c in df.columns]})

cols_summary = cols_summary[~cols_summary['Column'].isin(ELEMENTS)].reset_index(drop=True)

el_stats = pd.DataFrame({
    'Records with element (>0)': (el_num > 0).sum(),
    'Zero (element absent)': (el_num == 0).sum(),
    'Empty': el_num.isna().sum(),
    'Min (>0), at.%': el_num.where(el_num > 0).min(),
    'Median (>0), at.%': el_num.where(el_num > 0).median(),
    'Max, at.%': el_num.max()}).round(2)
el_stats['Share of records, %'] = (el_stats['Records with element (>0)'] / len(df) * 100).round(1)

proc_stats = pd.DataFrame({
    'Value > 0 (step applied)': (pr_num > 0).sum(),
    'Explicit 0': (pr_num == 0).sum(),
    'Empty (set to 0)': pr_num.isna().sum()})

opt_stats = pd.DataFrame({
    'Reported': df[NAN_COLS].notna().sum(),
    'Not reported (NaN)': df[NAN_COLS].isna().sum(),
    'Zero values': (opt_num == 0).sum()})
opt_stats['Reported, %'] = (opt_stats['Reported'] / len(df) * 100).round(1)

cooling_vc = (pd.concat({c: df[c].astype(object).fillna('NaN').astype(str).value_counts()
                         for c in NAN_COLS_TXT}, axis=1)
              .fillna(0).astype(int).rename_axis('Value').reset_index())

phase_combo = df['Phase composition'].value_counts().rename_axis('Phase composition').reset_index(name='Records')
phase_tokens = (df['Phase composition'].str.split(' + ', regex=False).explode()
                .value_counts().rename_axis('Phase').reset_index(name='Records'))
flags = (pd.DataFrame({'Target state': df['Target state'].value_counts(dropna=False),
                       'L12 presence': df['L12 presence'].value_counts(dropna=False)})
         .fillna(0).astype(int).rename_axis('Value').reset_index())

In [5]:
# %%
# Заполнение
df[ELEMENTS] = el_num.fillna(0.0)
df[ZERO_COLS] = pr_num.fillna(0)
assert not df[ELEMENTS + ZERO_COLS].isna().any().any()

# Проверка суммы концентраций
TOLERANCE = 0.5
df['computed_sum'] = df[ELEMENTS].sum(axis=1)
dev = (df['computed_sum'] - 100.0).abs()
print(f"Строк с |sum - 100| > {TOLERANCE}: {int((dev > TOLERANCE).sum())}; "
      f"максимальное отклонение: {dev.max():.3f} ат.%")
if 'Sum' in df.columns:
    print(f"max |Sum из базы - computed_sum| = "
          f"{(pd.to_numeric(df['Sum'], errors='coerce') - df['computed_sum']).abs().max():.3f}")

# Файл с видимым текстом "NaN" (в памяти остаются настоящие NaN)
df_out = df.drop(columns=['computed_sum']).copy()
df_out[NAN_COLS] = df_out[NAN_COLS].astype(object).fillna("NaN")
df_out.to_excel(FILLED_PATH, index=False)
print(f"Сохранено: {FILLED_PATH} | {df_out.shape[0]} строк, {df_out.shape[1]} колонок")

# %%
# Экспорт статистики для таблиц в статье
sheets = {
    'Summary': summary,
    'Columns': cols_summary,
    'Elements': el_stats.rename_axis('Element').reset_index(),
    'Processing_params': proc_stats.rename_axis('Column').reset_index(),
    'Optional_fields': opt_stats.rename_axis('Column').reset_index(),
    'Cooling_values': cooling_vc,
    'Phase_combinations': phase_combo,
    'Phase_individual': phase_tokens,
    'Target_L12_flags': flags,
}
with pd.ExcelWriter(STATS_PATH, engine='openpyxl') as writer:
    for name, table in sheets.items():
        table.to_excel(writer, sheet_name=name, index=False)
        ws = writer.sheets[name]
        for col_cells in ws.columns:
            width = max(len(str(c.value or '')) for c in col_cells)
            ws.column_dimensions[col_cells[0].column_letter].width = min(max(width + 3, 10), 60)
print(f"Статистика сохранена: {STATS_PATH}")


Строк с |sum - 100| > 0.5: 0; максимальное отклонение: 0.200 ат.%
max |Sum из базы - computed_sum| = 0.000
Сохранено: C:\Users\Николай\PycharmProjects\alloy-space-wip\data\processed\Database-Fe-Ni-Co-Al-Cr-Ti_filled.xlsx | 374 строк, 38 колонок
Статистика сохранена: C:\Users\Николай\PycharmProjects\alloy-space-wip\outputs\tables\data_audit_statistics.xlsx


In [6]:
# %%
# Выходные файлы
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# %%
# 6. Флаги фаз по множеству фаз в метке
PHASE_FLAGS = {'FCC': 'FCC', 'L12': 'L12', 'B2': 'B2', 'BCC': 'BCC',
               'eta': 'η', 'sigma': 'σ', 'L21': 'L21', 'Laves': 'Laves'}

phase_sets = df['Phase composition'].str.split(' + ', regex=False).map(frozenset)
known = set(PHASE_FLAGS.values()) | {'IM'}
print("Фазы без отдельного флага:", sorted(set().union(*phase_sets) - known))

for key, token in PHASE_FLAGS.items():
    df[f'phase_{key}'] = phase_sets.map(lambda s, t=token: int(t in s))
FLAG_COLS = [f'phase_{k}' for k in PHASE_FLAGS]

df['Target state'] = pd.to_numeric(df['Target state'], errors='raise').astype(int)
df_full = df.drop(columns=['computed_sum'])

# %%
# 7. Уникальные составы (ключ — концентрации, округлённые до 3 знаков)
df_key = df_full.copy()
df_key[ELEMENTS] = df_full[ELEMENTS].round(3)

n_exact = df_full.groupby(ELEMENTS).ngroups
n_rounded = df_key.groupby(ELEMENTS).ngroups
print(f"Уникальных составов: точное сравнение {n_exact}, после округления до 3 знаков {n_rounded}")

agg_spec = {
    'Composition (at.%)': ('Composition (at.%)', 'first'),
    'papers_count': ('DOI', 'nunique'),
    'n_records': ('DOI', 'size'),
    'Target state': ('Target state', 'max'),
    **{c: (c, 'max') for c in FLAG_COLS},
}
df_unique = df_key.groupby(ELEMENTS, as_index=False).agg(**agg_spec)

conflicts = df_key.groupby(ELEMENTS)['Phase composition'].nunique()
print(f"Записей: {len(df_full)}, уникальных составов: {len(df_unique)}")
print(f"Составов с разным фазовым составом при разных обработках: {int((conflicts > 1).sum())}")

# %%
# 8. Сохранение
df_full.to_parquet(PROCESSED_DIR / "dataset_experimental_full.parquet", index=False)
df_unique.to_parquet(PROCESSED_DIR / "dataset_compositions_unique.parquet", index=False)
print("Сохранено: dataset_experimental_full.parquet, dataset_compositions_unique.parquet")

Фазы без отдельного флага: ['Al-rich phase', 'Cr23C6', 'D019', 'D022', 'borides', 'carbides', 'other', 'τ']
Уникальных составов: точное сравнение 161, после округления до 3 знаков 161
Записей: 374, уникальных составов: 161
Составов с разным фазовым составом при разных обработках: 25
Сохранено: dataset_experimental_full.parquet, dataset_compositions_unique.parquet
